In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Read control table
# control_df = spark.sql("""
# SELECT *
# FROM iics.casemix.json_control_test
# WHERE is_active = 1
# """)
# checkpoint_path = "/Volumes/iics/casemix/history/json_stress_test_autoloader_checkpoints/"

In [0]:
control_df = spark.sql("""
SELECT *
FROM iics.casemix.json_control_test
WHERE s_no = 2
""")
checkpoint_path = "/Volumes/iics/casemix/history/json_stress_test_1_gb_autoloader_checkpoints/"

In [0]:

control = control_df.first()

source_path = control["file_path"]
target_table = f"{control['tgt_catalog']}.{control['tgt_schema']}.{control['tgt_table']}"



In [0]:
# df = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "text")
#     .option("wholetext", "true")
#     .load(source_path)
# )

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .option("wholetext", "true")
    .option("cloudFiles.maxFilesPerTrigger", "10000")   
    .option("cloudFiles.maxBytesPerTrigger", "20g")
    .load(source_path)
)

In [0]:
# df_final = (
#     df.select(
#         col("_metadata.file_path").alias("file_path"),

#         regexp_extract(
#             col("_metadata.file_path"),
#             r"([^/]+$)",
#             1
#         ).alias("file_name"),

#         col("value").alias("json_original_data"),

#         parse_json(col("value")).alias("json_data_for_parsing"),

#         round(col("_metadata.file_size") / 1024,2).alias("file_size_in_kb"),
        
#         col("_metadata.file_modification_time")
#             .cast("timestamp")
#             .alias("file_modified_date"),

#         regexp_extract(
#                     col("_metadata.file_path"),
#                     r".*_\d{8}_\d{8}_(\d{8})_\d{6}\.json$",
#                     1
#                 ).alias("dt_datafile"),

#         current_timestamp().alias("loaded_at")
#     )
# )

In [0]:
df_final = (
    df.select(
        col("_metadata.file_path").alias("file_path"),

        regexp_extract(
            col("_metadata.file_path"),
            r"([^/]+$)",
            1
        ).alias("file_name"),

        col("value").alias("json_original_data"),

        parse_json(col("value")).alias("json_data_for_parsing"),

        round(col("_metadata.file_size") / 1024, 2).alias("file_size_in_kb"),

        col("_metadata.file_modification_time")
            .cast("timestamp")
            .alias("file_modified_date"),

        regexp_extract(
            col("_metadata.file_path"),
            r".*_\d{8}_\d{8}_(\d{8})_\d{6}\.json$",
            1
        ).alias("dt_datafile"),

        current_timestamp().alias("loaded_at")
    )
    .repartition(128)       # Tune based on cluster size
)

In [0]:
# (
#     df_final.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", checkpoint_path)
#     .trigger(availableNow=True)
#     .toTable(target_table)
# )

In [0]:
query = (
    df_final.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
# %sql
# SELECT *
# FROM iics.casemix.json_control_test
# WHERE s_no = 3